# Dataset EDA (read-only)

**What it does**

- Runs exploratory data analysis on one YOLO-format dataset: image/label pairing checks, annotation counts per class, annotations per image, image resolutions, images below the tile size, and box sizes in pixels per class.
- Only reads. It never copies, edits or deletes anything, so it is safe to point at raw data.
- Cleaning itself now lives in two scripts: `scripts/clean_to_x_classes.py` (keep chosen classes, drop images left without annotations) and `scripts/filter_low_res.py` (drop images below a minimum resolution).

**How to use**

1. Set `DATASET_DIR` in the config cell below to any dataset root with `images/` and `labels/`: raw (`../data/data_raw/<name>`), cleaned (`../data/data-cleaned/<name>`), 5-defect or tiled.
2. For other layouts (e.g. Roboflow's `train/images`), override `IMAGES_DIR`, `LABELS_DIR` and `CLASSES_PATH` in the same cell.
3. Set `TILE_SIZE` to the tile size you plan to use (default 2048).
4. Run all cells, from the `data-cleaning/` folder. Typical use: once on the raw dataset before cleaning, and once on the cleaned output.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from PIL import Image

sys.path.insert(0, "scripts")
from clean_to_x_classes import load_source_classes  # reads classes.txt, data.yaml or notes.json

# ---- Configure per run: point this at the dataset you want to inspect ----
DATASET_DIR = Path("../data/data_raw/tf_r2")  # raw, cleaned, 5-defect or tiled dataset root
TILE_SIZE = 2048  # images with a shorter side below this produce no tiles

IMAGES_DIR = DATASET_DIR / "images"
LABELS_DIR = DATASET_DIR / "labels"
CLASSES_PATH = None  # None = first of classes.txt, data.yaml, notes.json found in DATASET_DIR
# ---------------------------------------------------------------------------

if CLASSES_PATH is None:
    candidates = [DATASET_DIR / name for name in ("classes.txt", "data.yaml", "notes.json")]
    CLASSES_PATH = next((p for p in candidates if p.exists()), None)
    if CLASSES_PATH is None:
        raise FileNotFoundError(f"No classes.txt, data.yaml or notes.json in {DATASET_DIR}; set CLASSES_PATH.")

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
class_names = load_source_classes(Path(CLASSES_PATH))  # {class id: name}
print(f"Dataset : {DATASET_DIR}")
print(f"Classes : {CLASSES_PATH} ({len(class_names)} classes)")

## 1) Image-label pairing checks

In [ ]:
image_files = sorted(p for p in IMAGES_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
label_files = sorted(LABELS_DIR.rglob("*.txt"))

image_ids = {p.stem for p in image_files}
label_ids = {p.stem for p in label_files}

images_missing_label = sorted(image_ids - label_ids)
labels_missing_image = sorted(label_ids - image_ids)

pd.DataFrame({
    "item": ["total_image_files", "total_label_files", "images_missing_label", "labels_missing_image"],
    "value": [len(image_files), len(label_files), len(images_missing_label), len(labels_missing_image)],
})

In [ ]:
pd.DataFrame({"image_id": images_missing_label})

In [ ]:
pd.DataFrame({"label_id": labels_missing_image})

## 2) Image sizes and annotations

Reads every image header and label file once; the sections below all use these two tables.

In [ ]:
label_by_id = {p.stem: p for p in label_files}

image_rows = []
box_rows = []
bad_rows = 0
unknown_class_rows = 0

for image_path in image_files:
    with Image.open(image_path) as img:
        width, height = img.size

    num_annotations = 0
    label_path = label_by_id.get(image_path.stem)
    if label_path is not None:
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            try:
                class_id = int(parts[0])
                box_w, box_h = float(parts[3]) * width, float(parts[4]) * height
            except (ValueError, IndexError):
                bad_rows += 1
                continue
            if class_id not in class_names:
                unknown_class_rows += 1
                continue
            num_annotations += 1
            box_rows.append({
                "image_id": image_path.stem,
                "class_id": class_id,
                "class_name": class_names[class_id],
                "box_w_px": box_w,
                "box_h_px": box_h,
                "long_side_px": max(box_w, box_h),
            })

    image_rows.append({
        "image_id": image_path.stem,
        "width": width,
        "height": height,
        "short_side": min(width, height),
        "num_annotations": num_annotations,
    })

images_df = pd.DataFrame(image_rows, columns=["image_id", "width", "height", "short_side", "num_annotations"])
boxes_df = pd.DataFrame(box_rows, columns=["image_id", "class_id", "class_name", "box_w_px", "box_h_px", "long_side_px"])

pd.DataFrame({
    "item": ["total_annotations", "images_without_annotations", "malformed_label_rows", "rows_with_class_id_not_in_class_list"],
    "value": [len(boxes_df), int((images_df["num_annotations"] == 0).sum()), bad_rows, unknown_class_rows],
})

### a) Per-class annotation counts

In [ ]:
per_class_df = pd.DataFrame(
    [
        {
            "class_id": class_id,
            "class_name": name,
            "annotation_count": int((boxes_df["class_id"] == class_id).sum()),
            "images_with_class": boxes_df.loc[boxes_df["class_id"] == class_id, "image_id"].nunique(),
        }
        for class_id, name in sorted(class_names.items())
    ]
)
per_class_df["percent_of_total"] = (per_class_df["annotation_count"] / max(len(boxes_df), 1) * 100).round(2)
per_class_df.sort_values("annotation_count", ascending=False).reset_index(drop=True)

### b) Distribution of annotations per image

In [ ]:
distribution = images_df["num_annotations"].value_counts().sort_index().reset_index()
distribution.columns = ["annotation_count", "num_images"]
distribution

### c) Image resolution breakdown

In [ ]:
resolution_df = (
    images_df.assign(resolution=images_df["width"].astype(str) + "x" + images_df["height"].astype(str))
    .groupby("resolution")
    .agg(num_images=("image_id", "count"), short_side=("short_side", "first"))
    .reset_index()
    .sort_values(["num_images", "resolution"], ascending=[False, True])
    .reset_index(drop=True)
)
resolution_df

### d) Images below the tile size

`tile_building_defect_images.py` skips any image whose shorter side is below the tile size, and `filter_low_res.py --min-side TILE_SIZE` removes the same images. This shows what that costs.

In [ ]:
below = images_df[images_df["short_side"] < TILE_SIZE]

pd.DataFrame({
    "item": [
        f"images_below_{TILE_SIZE}",
        f"images_at_or_above_{TILE_SIZE}",
        "annotations_on_images_below",
        "percent_of_images_below",
    ],
    "value": [
        len(below),
        len(images_df) - len(below),
        int(below["num_annotations"].sum()),
        f"{len(below) / max(len(images_df), 1) * 100:.2f}%",
    ],
})

In [ ]:
below.sort_values("short_side").reset_index(drop=True)

### e) Box size in pixels per class

Long side of each box in source-image pixels. A tile of `TILE_SIZE` is rescaled to the 640 px training input, so `median_at_640` is roughly how large the median object is when the model sees it.

In [ ]:
box_size_df = (
    boxes_df.groupby("class_name")["long_side_px"]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    .drop(columns=["mean", "std"])
    .rename(columns={"50%": "median"})
    .round(0)
)
box_size_df["median_at_640"] = (box_size_df["median"] * 640 / TILE_SIZE).round(1)
box_size_df["boxes_longer_than_tile"] = boxes_df[boxes_df["long_side_px"] > TILE_SIZE].groupby("class_name").size()
box_size_df["boxes_longer_than_tile"] = box_size_df["boxes_longer_than_tile"].fillna(0).astype(int)
box_size_df.sort_values("count", ascending=False)